# Chapter 2: Dataset Profile and Exploratory Data Analysis
### MATH 4230 Capstone Project
**Dataset:** Student Lifestyle, Mental Health and Burnout Insight

---

This chapter introduces the dataset, draws a working sample, profiles every variable, examines distributions and correlations, defines the train / validation / test split that will be used for the rest of the report, and saves a reusable preprocessing pipeline plus the processed arrays that every downstream chapter will load.

**Targets**
- Regression: `burnout_score` (continuous, 0 to 10)
- Classification: `risk_level` (Low = 0, High = 1)

**Working sample:** 200,000 rows stratified on `risk_level` (drawn from the full 1,000,000 row source).

---
## Setup

In [4]:
# Imports
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import joblib
import os
import warnings

warnings.filterwarnings('ignore')

# Plot defaults
plt.rcParams['figure.dpi'] = 100
plt.rcParams['savefig.dpi'] = 150
plt.rcParams['axes.spines.top'] = False
plt.rcParams['axes.spines.right'] = False
sns.set_style('whitegrid')

# Reproducibility
RANDOM_STATE = 230
np.random.seed(RANDOM_STATE)

# Paths
DATA_PATH     = '../Datasets/student_mental_health_burnout_1M.csv'
FIGURES_DIR   = '../figures/ch02/'
RESULTS_DIR   = '../results/ch02/'
ARTIFACTS_DIR = '../results/'

for d in (FIGURES_DIR, RESULTS_DIR, ARTIFACTS_DIR):
    os.makedirs(d, exist_ok=True)

print('Setup complete.')
print(f'RANDOM_STATE = {RANDOM_STATE}')

ModuleNotFoundError: No module named 'numpy'

---
## 2.1: Source and Provenance

The dataset, *Student Lifestyle, Mental Health and Burnout Insight*, is a publicly available collection of survey-style records on undergraduate students. Each row corresponds to one student and bundles together lifestyle indicators (study hours, sleep, screen time, social media use, exercise, diet quality), academic load (course load, GPA), mental health indicators (stress, anxiety, depression), and the two response variables used in this report: a continuous `burnout_score` from 0 to 10 and a binary `risk_level` flag (Low or High). The full file contains 1,000,000 rows and 20 columns. The download URL and date should be filled in below by the student running the notebook.

In [ ]:
# Provenance (fill in for your own run)
SOURCE_URL    = 'https://www.kaggle.com/datasets/ayeshasiddiqa123/student-health'
DOWNLOAD_DATE = '2026-05-09'

print(f'Source URL:    {SOURCE_URL}')
print(f'Downloaded on: {DOWNLOAD_DATE}')

# Load full dataset
df_full = pd.read_csv(DATA_PATH)
print(f'\nFull dataset shape: {df_full.shape}')
print(f'  rows:    {df_full.shape[0]:,}')
print(f'  columns: {df_full.shape[1]}')

print('\nColumn dtypes:')
print(df_full.dtypes.to_string())

---
## 2.2: Data Dictionary

The table below lists every variable, its data type as stored, its measurement scale, and whether it is a predictor or a target. The dictionary is the contract between this chapter and every chapter that follows. Update the entries below if any column name differs from the version of the dataset actually loaded.

In [ ]:
# Build the data dictionary
data_dict = pd.DataFrame([
    # Demographics
    ('age',                  'int',     'years',                 'predictor'),
    ('gender',               'object',  'Male / Female',         'predictor (categorical)'),
    ('academic_year',        'int',     '1 to 4',                'predictor'),
    # Lifestyle
    ('study_hours_per_day',  'float',   'hours / day',           'predictor'),
    ('sleep_hours',          'float',   'hours / night',         'predictor'),
    ('physical_activity',    'float',   'hours / week',          'predictor'),
    ('screen_time',          'float',   'hours / day',           'predictor'),
    ('internet_usage',       'float',   'hours / day',           'predictor'),
    # Academic pressure
    ('exam_pressure',        'float',   '0 to 10',               'predictor'),
    ('academic_performance', 'float',   '0 to 10 or GPA-like',   'predictor'),
    # Mental health indicators
    ('stress_level',         'float',   '0 to 10',               'predictor'),
    ('anxiety_score',        'float',   '0 to 10',               'predictor'),
    ('depression_score',     'float',   '0 to 10',               'predictor'),
    ('social_support',       'float',   '0 to 10',               'predictor'),
    # Social / financial context
    ('financial_stress',     'float',   '0 to 10',               'predictor'),
    ('family_expectation',   'float',   '0 to 10',               'predictor'),
    # Auxiliary outcomes (treated as targets to drop from X)
    ('mental_health_index',  'float',   '0 to 10',               'target (auxiliary)'),
    ('dropout_risk',         'float',   '0 to 1 probability',    'target (auxiliary)'),
    # Primary targets
    ('burnout_score',        'float',   '0 to 10',               'target (regression)'),
    ('risk_level',           'object',  'Low / High',            'target (classification)'),
], columns=['Variable', 'Type', 'Units / Scale', 'Role'])

# Filter the dictionary to only the columns that actually appear in the loaded file
# (and report any mismatch so it can be fixed at the top of the chapter)
present  = [v for v in data_dict['Variable'] if v in df_full.columns]
missing  = [v for v in data_dict['Variable'] if v not in df_full.columns]
extra    = [c for c in df_full.columns        if c not in data_dict['Variable'].values]

if missing:
    print(f'Listed in dictionary but NOT in CSV: {missing}')
if extra:
    print(f'In CSV but NOT in dictionary:        {extra}')

data_dict = data_dict[data_dict['Variable'].isin(df_full.columns)].reset_index(drop=True)
data_dict

**Reading the table.** Variables marked *predictor* enter the feature matrix `X`. The two primary targets (`burnout_score`, `risk_level`) are held out of `X` in the preprocessing pipeline. The two auxiliary outcomes (`mental_health_index`, `dropout_risk`) are also dropped from `X` because they encode the same underlying construct as the primary targets and would cause leakage if left in.

---
## 2.3: Sampling

The full file contains one million rows, which is unnecessarily large for the methods in this report and slows every cross-validation loop. A 200,000-row sample is drawn instead, stratified on `risk_level` so the class balance in the sample matches the population balance exactly. This sample becomes the working dataset for the remainder of the project.

In [ ]:
SAMPLE_SIZE = 200_000

# Stratified sample on risk_level
sample_idx, _ = train_test_split(
    np.arange(len(df_full)),
    train_size=SAMPLE_SIZE,
    stratify=df_full['risk_level'],
    random_state=RANDOM_STATE
)

df = df_full.iloc[sample_idx].reset_index(drop=True)

print(f'Sample shape:       {df.shape}')
print(f'Percent of full:    {100 * len(df) / len(df_full):.1f}%')

print('\nClass balance of risk_level in sample:')
balance = df['risk_level'].value_counts(normalize=True).sort_index()
for k, v in balance.items():
    print(f'  {k:6s}  {v:.4f}  ({int(v * len(df)):,} rows)')

# Free memory: we no longer need the full file
del df_full

---
## 2.4: Summary Statistics

 The table is also written to `summary_stats.csv`.

In [ ]:
numeric_cols = sorted(df.select_dtypes(include=np.number).columns.tolist())

rows = []
for col in numeric_cols:
    s = df[col]
    rows.append({
        'Variable':  col,
        'n':         int(s.notna().sum()),
        'Mean':      int(round(s.mean())),
        'Std Dev':   round(s.std(), 3),
        'Median':    round(s.median(), 3),
        'Q1':        round(s.quantile(0.25), 3),
        'Q3':        round(s.quantile(0.75), 3),
        'Min':       round(s.min(), 3),
        'Max':       round(s.max(), 3),
        'Missing %': round(100 * s.isna().mean(), 3),
    })

summary_stats = pd.DataFrame(rows)
summary_stats.to_csv(os.path.join(RESULTS_DIR, 'summary_stats.csv'), index=False)

print(f'Saved: {os.path.join(RESULTS_DIR, "summary_stats.csv")}')
summary_stats

**Note.** Means are rounded to the nearest integer in this table per the chapter rubric. The unrounded means are still used inside any downstream calculation (the rounding here is for display only).

---
## 2.5: Missing Data and Outliers

In [ ]:
# Missing values
missing = df.isna().sum()
missing = missing[missing > 0].sort_values(ascending=False)

print('Missing value counts (columns with at least one missing value):')
if len(missing) == 0:
    print('  none')
else:
    for col, n in missing.items():
        print(f'  {col:30s}  {n:>8,}  ({100 * n / len(df):.3f}%)')

In [ ]:
# Outliers via z-score, threshold 3.5
Z_THRESHOLD = 3.5

outlier_rows = []
for col in numeric_cols:
    s = df[col].dropna()
    if s.std() == 0:
        n_out = 0
    else:
        z = np.abs(stats.zscore(s))
        n_out = int((z > Z_THRESHOLD).sum())
    outlier_rows.append((col, n_out, round(100 * n_out / len(s), 3)))

outlier_df = pd.DataFrame(outlier_rows, columns=['Variable', 'Outliers', '% of column'])
outlier_df = outlier_df.sort_values('Outliers', ascending=False).reset_index(drop=True)

print(f'Outliers per numeric column (|z| > {Z_THRESHOLD}):')
print(outlier_df.to_string(index=False))

**Missing data strategy.** Missing values are rare in this dataset. The chosen approach is to retain rows with at most one missing predictor and impute them with the training-set median inside the preprocessing pipeline; rows with multiple missing predictors are dropped. If the printout above shows zero missing values, no imputation is performed and the strategy is moot.

**Outlier strategy.** Outliers are kept rather than removed. The z-score threshold of 3.5 flags candidates for inspection but does not trigger deletion: nothing in the variable definitions suggests that extreme study-hour, screen-time, or stress values are data errors, and tree-based methods (used in several later chapters) handle them naturally.

---
## 2.6: Distributions

In [ ]:
# Identify predictor columns (numeric only) by excluding targets and auxiliary targets
target_cols = [c for c in ['burnout_score', 'risk_level', 'mental_health_index', 'dropout_risk']
               if c in df.columns]

predictor_numeric = [c for c in numeric_cols if c not in target_cols]

print(f'{len(predictor_numeric)} numeric predictors will be plotted.')

In [ ]:
# Histograms for numeric predictors
# This version saves 6 histograms per image so the labels are easier to read in Word.

plots_per_fig = 6
ncols = 3
nrows = 2

for fig_num, start in enumerate(range(0, len(predictor_numeric), plots_per_fig), start=1):
    
    # Select 6 variables at a time
    cols_subset = predictor_numeric[start:start + plots_per_fig]
    
    fig, axes = plt.subplots(nrows, ncols, figsize=(15, 7))
    axes = np.array(axes).reshape(-1)

    for i, col in enumerate(cols_subset):
        ax = axes[i]
        ax.hist(
            df[col].dropna(),
            bins=40,
            color='#4C72B0',
            edgecolor='white',
            linewidth=0.4
        )
        ax.set_title(col, fontsize=12)
        ax.set_xlabel(col, fontsize=10)
        ax.set_ylabel('count', fontsize=10)
        ax.tick_params(axis='both', labelsize=9)

    # Turn off unused axes if the last figure has fewer than 6 plots
    for j in range(len(cols_subset), len(axes)):
        axes[j].axis('off')

    fig.suptitle(
        f'Figure 2.1.{fig_num}: Distributions of Numeric Predictors',
        fontsize=15,
        y=1.02
    )

    plt.tight_layout()

    out = os.path.join(
        FIGURES_DIR,
        f'predictor_distributions_part_{fig_num}.png'
    )

    plt.savefig(out, dpi=200, bbox_inches='tight')
    plt.show()

    print(f'Saved: {out}')

In [ ]:
# Bar charts for categorical variables: risk_level always, gender if non-numeric
cat_cols = ['risk_level']
if 'gender' in df.columns and not pd.api.types.is_numeric_dtype(df['gender']):
    cat_cols.append('gender')

fig, axes = plt.subplots(1, len(cat_cols), figsize=(5.5 * len(cat_cols), 4))
if len(cat_cols) == 1:
    axes = [axes]

for ax, col in zip(axes, cat_cols):
    counts = df[col].value_counts().sort_index()
    bars = ax.bar(counts.index.astype(str), counts.values,
                  color=['#4C72B0', '#DD8452', '#55A868'][:len(counts)],
                  edgecolor='white', linewidth=0.5)
    ax.set_title(col)
    ax.set_xlabel('')
    ax.set_ylabel('count')
    for bar, v in zip(bars, counts.values):
        ax.text(bar.get_x() + bar.get_width() / 2, v,
                f'{v:,}', ha='center', va='bottom', fontsize=9)

fig.suptitle('Figure 2.2: Distributions of Categorical Variables', fontsize=13, y=1.02)
plt.tight_layout()
out = os.path.join(FIGURES_DIR, 'categorical_distributions.png')
plt.savefig(out, dpi=150, bbox_inches='tight')
plt.show()
print(f'Saved: {out}')

In [ ]:
# Histogram for the regression target
fig, ax = plt.subplots(figsize=(9, 4.5))
ax.hist(df['burnout_score'].dropna(), bins=50, color='#C44E52', edgecolor='white', linewidth=0.4)
ax.axvline(df['burnout_score'].mean(),   color='black', linestyle='--', linewidth=1, label=f"mean = {df['burnout_score'].mean():.2f}")
ax.axvline(df['burnout_score'].median(), color='black', linestyle=':',  linewidth=1, label=f"median = {df['burnout_score'].median():.2f}")
ax.set_title('Figure 2.3: Distribution of burnout_score (Regression Target)', fontsize=12)
ax.set_xlabel('burnout_score')
ax.set_ylabel('count')
ax.legend()
plt.tight_layout()
out = os.path.join(FIGURES_DIR, 'target_distribution.png')
plt.savefig(out, dpi=150, bbox_inches='tight')
plt.show()
print(f'Saved: {out}')

---
## 2.7: Correlation Matrix

The heatmap below shows Pearson correlations among every numeric variable in the sample, predictors and targets together. Pairs with absolute correlation above 0.60 are printed beneath the figure. These are the candidates worth watching when multicollinearity becomes an issue in Chapters 4, 7, and 10.

In [ ]:
corr = df[numeric_cols].corr()

fig, ax = plt.subplots(figsize=(11, 9))
mask = np.triu(np.ones_like(corr, dtype=bool), k=1)
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='RdBu_r',
            center=0, vmin=-1, vmax=1, square=True, linewidths=0.5,
            cbar_kws={'shrink': 0.7, 'label': 'Pearson r'},
            annot_kws={'size': 8}, ax=ax)
ax.set_title('Figure 2.4: Correlation Matrix of Numeric Variables', fontsize=13, pad=12)
ax.set_xticklabels(ax.get_xticklabels(), rotation=45, ha='right'),
plt.tight_layout()
out = os.path.join(FIGURES_DIR, 'correlation_heatmap.png')
plt.savefig(out, dpi=150, bbox_inches='tight')
plt.show()
print(f'Saved: {out}')

In [ ]:
# Pairs with |r| > 0.60
THRESHOLD = 0.60

pairs = []
cols = corr.columns.tolist()
for i in range(len(cols)):
    for j in range(i + 1, len(cols)):
        r = corr.iloc[i, j]
        if abs(r) > THRESHOLD:
            pairs.append((cols[i], cols[j], round(r, 3)))

pairs.sort(key=lambda x: -abs(x[2]))

print(f'Variable pairs with |r| > {THRESHOLD}:')
if not pairs:
    print('  none')
else:
    for a, b, r in pairs:
        print(f'  {a:30s}  {b:30s}  r = {r:+.3f}')

---
## 2.8: Train / Validation / Test Split

The sample is split 70 / 10 / 20 into training, validation, and test sets, stratified on `risk_level` so all three sets share the same class balance.

> **Convention used for the rest of the report.** Chapters 3 through 12 use only the training set (70%) for fitting and the test set (20%) for final evaluation. The validation set (10%) is held aside and is touched only in Chapter 13 (Neural Networks), where it is used for early stopping and hyperparameter selection.

In [ ]:
# 70 / 10 / 20 split, stratified on risk_level
# Step 1: split off the test set (20%)
df_trainval, df_test = train_test_split(
    df, test_size=0.20,
    stratify=df['risk_level'],
    random_state=RANDOM_STATE
)

# Step 2: split the remaining 80% into 70 / 10  (val = 10 / 80 = 0.125 of the 80%)
df_train, df_val = train_test_split(
    df_trainval, test_size=0.125,
    stratify=df_trainval['risk_level'],
    random_state=RANDOM_STATE
)

df_train = df_train.reset_index(drop=True)
df_val   = df_val.reset_index(drop=True)
df_test  = df_test.reset_index(drop=True)

# Report sizes and class balance
def split_summary(name, frame):
    n = len(frame)
    bal = frame['risk_level'].value_counts(normalize=True).sort_index()
    bal_str = '   '.join(f'{k} = {v:.4f}' for k, v in bal.items())
    print(f'  {name:5s}  n = {n:>7,}  ({100 * n / len(df):5.2f}%)   {bal_str}')

print('Split sizes and class balance:')
split_summary('train', df_train)
split_summary('val',   df_val)
split_summary('test',  df_test)
print(f'\nTotal: {len(df_train) + len(df_val) + len(df_test):,} rows')

---
## 2.9: Preprocessing Pipeline

The function below is the single source of truth for how raw rows become feature matrices. It does four things:

1. Drops the four target / auxiliary-target columns from the feature side: `burnout_score`, `risk_level`, `mental_health_index`, and `dropout_risk`. Dropping the auxiliary outcomes prevents target leakage, since these variables encode closely related student-risk and mental-health outcomes.
2. Dummy-encodes `gender` with `drop_first=True`. Since gender has three categories, two dummy variables are created and one category is used as the reference group.
3. Dummy-encodes `academic_year` with `drop_first=True`. Although it is stored as integers from 1 to 4, it is treated as categorical because the values represent year groups rather than a continuous measurement.
4. Encodes `risk_level` as a multiclass classification target: Low = 0, Medium = 1, High = 2.
5. Fits a `StandardScaler` on the training features only and applies it to train, validation, and test.

Calling `preprocess()` returns the design matrices `X_train`, `X_val`, `X_test`, both targets (`y_*_reg`, `y_*_cls`), the fitted scaler, and the feature names.

In [ ]:
TARGETS_TO_DROP = [
    'burnout_score',
    'risk_level',
    'mental_health_index',
    'dropout_risk'
]


def preprocess(df_train, df_val, df_test, random_state=RANDOM_STATE):
    """
    Apply the chapter-2 preprocessing pipeline.

    This function prepares the same input structure for every later model.

    Steps:
      1. Separate the regression and classification targets.
      2. Drop target-related columns from the predictor matrix.
      3. Dummy-encode gender and academic_year with drop_first=True.
      4. Align validation and test columns to match the training columns.
      5. Convert all predictors to float.
      6. Fit StandardScaler on X_train only.
      7. Transform X_train, X_val, and X_test.

    Returns
    -------
    dict with keys:
        X_train, X_val, X_test
        y_train_reg, y_val_reg, y_test_reg
        y_train_cls, y_val_cls, y_test_cls
        scaler
        feature_names
    """

    # ------------------------------------------------------------
    # 1. Separate target variables
    # ------------------------------------------------------------

    # Regression target
    y_train_reg = df_train['burnout_score'].to_numpy()
    y_val_reg   = df_val['burnout_score'].to_numpy()
    y_test_reg  = df_test['burnout_score'].to_numpy()

    # Classification target
    # Low = 0, Medium = 1, High = 2
    cls_map = {
        'Low': 0,
        'Medium': 1,
        'High': 2
    }

    y_train_cls = df_train['risk_level'].map(cls_map).astype(int).to_numpy()
    y_val_cls   = df_val['risk_level'].map(cls_map).astype(int).to_numpy()
    y_test_cls  = df_test['risk_level'].map(cls_map).astype(int).to_numpy()


    # ------------------------------------------------------------
    # 2. Build predictor data frames
    # ------------------------------------------------------------

    # Only drop target-related columns that exist in the training data
    drop_cols = [
        c for c in TARGETS_TO_DROP
        if c in df_train.columns
    ]

    X_train_df = df_train.drop(columns=drop_cols).copy()
    X_val_df   = df_val.drop(columns=drop_cols).copy()
    X_test_df  = df_test.drop(columns=drop_cols).copy()


    # ------------------------------------------------------------
    # 3. Dummy-encode categorical predictors
    # ------------------------------------------------------------

    # gender has Female, Male, Other
    # academic_year has 1, 2, 3, 4 and is treated as categorical
    categorical_cols = [
        'gender',
        'academic_year'
    ]

    to_dummy = [
        c for c in categorical_cols
        if c in X_train_df.columns
    ]

    if to_dummy:
        X_train_df = pd.get_dummies(
            X_train_df,
            columns=to_dummy,
            drop_first=True
        )

        X_val_df = pd.get_dummies(
            X_val_df,
            columns=to_dummy,
            drop_first=True
        )

        X_test_df = pd.get_dummies(
            X_test_df,
            columns=to_dummy,
            drop_first=True
        )


    # ------------------------------------------------------------
    # 4. Align validation and test columns with training columns
    # ------------------------------------------------------------

    # Save training feature names after dummy encoding
    feature_names = X_train_df.columns.tolist()

    # Reindex validation and test sets so they have the same columns
    # in the same order as the training set
    X_val_df = X_val_df.reindex(
        columns=feature_names,
        fill_value=0
    )

    X_test_df = X_test_df.reindex(
        columns=feature_names,
        fill_value=0
    )


    # ------------------------------------------------------------
    # 5. Convert predictors to float
    # ------------------------------------------------------------

    # Dummy variables may be stored as Boolean values,
    # so all predictors are converted to float before scaling.
    X_train_df = X_train_df.astype(float)
    X_val_df   = X_val_df.astype(float)
    X_test_df  = X_test_df.astype(float)


    # ------------------------------------------------------------
    # 6. Standardize predictors
    # ------------------------------------------------------------

    # Fit the scaler on training data only to avoid data leakage.
    scaler = StandardScaler()

    X_train = scaler.fit_transform(
        X_train_df.to_numpy()
    )

    X_val = scaler.transform(
        X_val_df.to_numpy()
    )

    X_test = scaler.transform(
        X_test_df.to_numpy()
    )


    # ------------------------------------------------------------
    # 7. Return processed data
    # ------------------------------------------------------------

    return {
        'X_train': X_train,
        'X_val': X_val,
        'X_test': X_test,

        'y_train_reg': y_train_reg,
        'y_val_reg': y_val_reg,
        'y_test_reg': y_test_reg,

        'y_train_cls': y_train_cls,
        'y_val_cls': y_val_cls,
        'y_test_cls': y_test_cls,

        'scaler': scaler,
        'feature_names': feature_names
    }


# ------------------------------------------------------------
# Apply the preprocessing pipeline
# ------------------------------------------------------------

out = preprocess(
    df_train,
    df_val,
    df_test
)


# ------------------------------------------------------------
# Store processed feature matrices
# ------------------------------------------------------------

X_train = out['X_train']
X_val   = out['X_val']
X_test  = out['X_test']


# ------------------------------------------------------------
# Store regression targets
# ------------------------------------------------------------

y_train_reg = out['y_train_reg']
y_val_reg   = out['y_val_reg']
y_test_reg  = out['y_test_reg']


# ------------------------------------------------------------
# Store classification targets
# ------------------------------------------------------------

y_train_cls = out['y_train_cls']
y_val_cls   = out['y_val_cls']
y_test_cls  = out['y_test_cls']


# ------------------------------------------------------------
# Store fitted scaler and final feature names
# ------------------------------------------------------------

scaler = out['scaler']
feature_names = out['feature_names']


# ------------------------------------------------------------
# Confirm final shapes and feature names
# ------------------------------------------------------------

print(f'X_train shape: {X_train.shape}')
print(f'X_val   shape: {X_val.shape}')
print(f'X_test  shape: {X_test.shape}')

print(f'\n{len(feature_names)} feature(s):')

for i, name in enumerate(feature_names, 1):
    print(f'  {i:2d}. {name}')

---
## 2.10: Save Artifacts

Every processed array, the fitted scaler, and the per-split data frames are written to `../results/` (the shared artifacts directory). Each chapter loads from exactly these files, so reruns of any later chapter never need to re-execute the preprocessing pipeline.

In [ ]:
artifacts = {
    'X_train.npy':         X_train,
    'X_val.npy':           X_val,
    'X_test.npy':          X_test,
    'y_train_reg.npy':     y_train_reg,
    'y_val_reg.npy':       y_val_reg,
    'y_test_reg.npy':      y_test_reg,
    'y_train_cls.npy':     y_train_cls,
    'y_val_cls.npy':       y_val_cls,
    'y_test_cls.npy':      y_test_cls,
}

saved = []

# numpy arrays
for fname, arr in artifacts.items():
    path = os.path.join(ARTIFACTS_DIR, fname)
    np.save(path, arr)
    saved.append(path)

# scaler and feature names via joblib
scaler_path = os.path.join(ARTIFACTS_DIR, 'standard_scaler.pkl')
joblib.dump(scaler, scaler_path)
saved.append(scaler_path)

fn_path = os.path.join(ARTIFACTS_DIR, 'feature_names.pkl')
joblib.dump(feature_names, fn_path)
saved.append(fn_path)

# per-split data frames
for name, frame in [('df_train.csv', df_train),
                    ('df_val.csv',   df_val),
                    ('df_test.csv',  df_test)]:
    path = os.path.join(ARTIFACTS_DIR, name)
    frame.to_csv(path, index=False)
    saved.append(path)

# Report sizes
print(f'Saved {len(saved)} files to {ARTIFACTS_DIR}\n')
print(f'  {"file":30s} {"size":>12s}')
print(f'  {"-" * 30} {"-" * 12}')
for path in saved:
    size = os.path.getsize(path)
    if size < 1024:
        size_str = f'{size} B'
    elif size < 1024 ** 2:
        size_str = f'{size / 1024:.1f} KB'
    else:
        size_str = f'{size / 1024 ** 2:.2f} MB'
    print(f'  {os.path.basename(path):30s} {size_str:>12s}')

print('\nChapter 2 complete. Downstream chapters can now load these artifacts.')

---

**End of Chapter 2.**

Downstream chapters load artifacts with:

```python
import numpy as np
import joblib

X_train = np.load('../results/X_train.npy')
X_test  = np.load('../results/X_test.npy')
y_train_reg = np.load('../results/y_train_reg.npy')
y_test_reg  = np.load('../results/y_test_reg.npy')
y_train_cls = np.load('../results/y_train_cls.npy')
y_test_cls  = np.load('../results/y_test_cls.npy')

scaler         = joblib.load('../results/standard_scaler.pkl')
feature_names  = joblib.load('../results/feature_names.pkl')
```

The validation arrays (`X_val`, `y_val_reg`, `y_val_cls`) are loaded only in Chapter 13.